<a href="https://colab.research.google.com/github/Yahimeen/IA_AplicadaLLama_Hackathon_01/blob/main/Hackathon%20Asistente%20con%20Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **  HACKATHON -- ASISTENTE VIRTUAL **

Autor: **Pedro Enrique Ramirez Lopez**

Este Colab implementa un asistente virtual para demostrar los conceptos vistos en este curso: inferencia básica, RAG y fine-tuning con LoRa para construir un asistente que recupera contexto propio y responde con un modelo ya ajustado a un tono específico.

Para este asistente virtual se usa informacion de **mi curriculum** para proporcionar informacion inexistente en los modelos preentrenados para poder validar las respuestas.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

**Abriendo la sesion usando el token de Hugging Face**

In [1]:
# Instalar librerías e iniciar sesión en Hugging Face con el token desde Colab Secrets

# Remover los comentarios cuando se instala la primera vez
!pip install transformers peft accelerate trl sentence-transformers --quiet
!pip install PyMuPDF --quiet
!pip install chromadb --quiet
!pip install groq -q

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
from groq import Groq
from datasets import Dataset

logging.set_verbosity_error()

CHUNCK_SIZE=250
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
modelo_pequeno = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

pdf_file_path = "Resume Pedro Ramirez - Associated Director Digital Business Solutions - GT.pdf"

login(token=userdata.get('DEV_AI_READ_ONLY'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


Empezaremos por definir el base line del asistente, confirmando que usando solamente el conocimiento del modelo, no puede responder a las preguntas.

In [2]:
# funcion para poder usar el modelo de openai para hacer preguntas sin proporcionar informacion de contexto.
def asistente_virtual_sincontexto(pregunta):

    prompt = f"""Responde la pregunta.

              Pregunta: {pregunta}

              Proporciona una respuesta Concisa, no incluyas caracteres extraños en la respuesta."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", # Usar el modelo Groq para la generación
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

pregunta = "¿Cuál es la experiencia de Pedro Ramirez en administracion de proyectos?"

print(f"Pregunta: {pregunta}")
print(f"Respuesta: {asistente_virtual_sincontexto(pregunta)}\n")

Pregunta: ¿Cuál es la experiencia de Pedro Ramirez en administracion de proyectos?
Respuesta: Pedro Ramírez cuenta con 10 años de experiencia liderando proyectos en el sector tecnológico, aplicando metodologías ágiles y garantizando entregas a tiempo y dentro del presupuesto.



## **PASO 1: LA BASE DE CONOCIMIENTO (RAG)**

Primero vamos a crear una funcion para extraer el texto del archivo que usaremos como base de conocimiento.

Indexaremos esta base de conocimientos con sentence-transformer para poder utilizarla para recuperar el fragmento mas relevante para responder la pregunta y evitar alucionaciones al responder.



In [3]:
import pymupdf
import fitz

def extract_pdf_chunks(pdf_path, chunk_size=1000):
    """
      Esta funcion extrae el texto de un archivo y genera 'chunks' de tamaño definido por el usuario, el default es 1000 caracteres.
    """
    chunks_list = []
    try:
        document = fitz.open(pdf_path)
        current_text = ""
        for page_num in range(document.page_count):
            page = document.load_page(page_num)
            page_text = page.get_text()
            current_text += page_text

            # Agrega los chunks a una lista
            while len(current_text) >= chunk_size:
                chunks_list.append(current_text[:chunk_size])
                current_text = current_text[chunk_size:]

        # Agrega el texto faltante a la lista
        if current_text:
            chunks_list.append(current_text)

        document.close()
        return chunks_list
    except FileNotFoundError:
        print(f"Error: Archivo PDF no encontrado en : {pdf_path}")
        return []
    except Exception as e:
        print(f"Ocurrio un error: {e}")
        return []


Luego vamos a extraer la informacion del archivo que sera nuestra base de datos de conocimientos para el asistente.

In [4]:

all_chunks = list(extract_pdf_chunks(pdf_file_path, chunk_size=CHUNCK_SIZE))

print(f"Total chunks extracted: {len(all_chunks)}")
for i, chunk in enumerate(all_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print("\n")

Total chunks extracted: 57
--- Chunk 1 ---
Pedro Enrique Ramirez Lopez
90 S Regan Mead Cir, The Woodlands, TX, 77382
+1 (203) 300 2267 – yahimeen@gmail.com
Admissions Committee, Online Master of Science in Computer Science (OMSCS)
I am pleased to submit my application to the Online Master of 


--- Chunk 2 ---
Science in Computer Science (OMSCS) program
at the Georgia Institute of Technology. With more than two decades of experience in information
technology, software development, data analytics, automation, and digital transformation, I am seeking to
furt


--- Chunk 3 ---
her strengthen my computer science foundation while expanding my expertise in artificial intelligence
and advanced computing technologies.
Throughout my career, I have worked at the intersection of business and technology, leading initiatives
involvi


--- Chunk 4 ---
ng software development, data engineering, analytics, robotic process automation, process mining,
conversational AI, machine learning, and generative 


Luego vamos a usar los chuncks para crear los embeddings.


In [5]:
# Este codigo deberia ser de una sola vez para poder generar los embeddings

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer(model_name)
print("Modelo de embeddings cargado:", modelo_embeddings)

embeddings_documentos = modelo_embeddings.encode(all_chunks)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings cargado: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
Embeddings generados: (57, 384)



A continuacion, usaremos una base de datos vectorial (chromadb) para almacenar los embeddings.

Definimos una funcion para almacenar los embeddings, de manera similar a la generacion de los embeddings, esta parte del proyecto se deberia llamar solamente cada vez que se requiera actualizar la base de conocimientos.


In [6]:
import chromadb

def guardar_embeddings_en_chromadb(fragmentos, embeddings, nombre_coleccion='fragmentos_curriculum'):
    """
    Almacena fragmentos y sus embeddings en una colección de ChromaDB.
    """
    cliente = chromadb.Client()
    # Crea una nueva colección u obtiene una existente
    coleccion = cliente.get_or_create_collection(name=nombre_coleccion)

    # Genera IDs para cada fragmento (pueden ser números secuenciales simples)
    ids_fragmentos = [f"fragmento_{i}" for i in range(len(fragmentos))]

    coleccion.add(
        embeddings=embeddings.tolist(), # ChromaDB espera una lista de listas
        documents=fragmentos,
        ids=ids_fragmentos
    )
    print(f"Almacenados {len(fragmentos)} fragmentos en la colección de ChromaDB '{nombre_coleccion}'.")
    return coleccion

# Llama a la función para almacenar tus embeddings generados
coleccion_chroma = guardar_embeddings_en_chromadb(all_chunks, embeddings_documentos)
print(f"Número de elementos en la colección: {coleccion_chroma.count()}")

Almacenados 57 fragmentos en la colección de ChromaDB 'fragmentos_curriculum'.
Número de elementos en la colección: 57



Ahora que los embeddings están almacenados en ChromaDB, vamos a usar una funcion para hacer busquedas en la colección por similitud.


In [7]:
# Funcion para hacer busquedas en la colección de ChromaDB
def consultar_coleccion_chroma(texto_consulta, coleccion, n_resultados=2):
    embedding_consulta = modelo_embeddings.encode([texto_consulta]).tolist()
    resultados = coleccion.query(
        query_embeddings=embedding_consulta,
        n_results=n_resultados,
        include=['documents', 'distances']
    )
    return resultados

# Prueba la función de consulta
consulta = "¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?"
resultados_consulta = consultar_coleccion_chroma(consulta, coleccion_chroma, n_resultados=3)

print(f"Consulta: {consulta}")
print("Mejores resultados de ChromaDB:")
for i, (doc, dist) in enumerate(zip(resultados_consulta['documents'][0], resultados_consulta['distances'][0])):
    print(f"--- Resultado {i+1} (Distancia: {dist:.4f}) ---")
    print(doc)
    print("\n")

Consulta: ¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?
Mejores resultados de ChromaDB:
--- Resultado 1 (Distancia: 17.7462) ---
Post Graduate Diploma on Project Management (PMI),
06-2008 / 11-2008. Instituto Tecnologico Autonomo de Mexico. Certified.
Post Graduate Diploma on Development of Management Skills,
10-2003 / 02-2004. Instituto Tecnologico de Estudios Superiores de M


--- Resultado 2 (Distancia: 17.9947) ---
ption of Project Management tools.

Technology Strategy and Governance: Defined and maintained a regulatory framework for the
evaluation, selection, and implementation of technologies, ensuring alignment with Praxair
Mexico’s strategic platform goal


--- Resultado 3 (Distancia: 18.1956) ---
ive collaboration and driving successful project execution in
diverse environments.
Project Leader, Praxair Mexico
Dec 1999 – Jul 2008
Key Achievements and Responsibilities:

Business Intelligence Development: Implemented BI solutions utilizing MS S




## **PASO 2: AJUSTAR EL MODELO (FINE-TUNING CON LoRA)**

De acuerdo a los parametros solicitados para este proyecto, a continuacion se usara un modelo ligero basado en Llama para ajustarlo con LoRA para que responda siempre en el mismo tono breve y directo.
Usaremos el mismo modelo sugerido en el Colab de la sesion correspondiente.

In [8]:
# Cargar el modelo base y su tokenizer

tokenizer = AutoTokenizer.from_pretrained(modelo_pequeno)
modelo = AutoModelForCausalLM.from_pretrained(modelo_pequeno, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_pequeno)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Voy a definir una funcion que ayudara al asistente a responder cualquier pregunta usando la informacion de la base de datos vectorial

In [9]:
import time
from groq import Groq, APIStatusError # Importamos APIStatusError

def asistente_virtual_con_chromadb(pregunta):
    # Recuperar el fragmento más relevante de ChromaDB
    resultados_recuperados = consultar_coleccion_chroma(pregunta, coleccion_chroma, n_resultados=2)
    fragmento = resultados_recuperados['documents'][0][0] # Tomar el primer y más relevante fragmento

    prompt = f"""Responde la pregunta usando SOLO la informacion proporcionada a continuacion.
                 Si la informacion no responde la pregunta, no inventes ninguna respuesta, responde diciendo que no sabes la respuesta.

Informacion: {fragmento}

Pregunta: {pregunta}

Respuesta Concisa, sin caracteres extraños."""

    max_retries = 3
    retry_delay = 10 # segundos
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b", # Usar el modelo Groq para la generación
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except APIStatusError as e:
            if e.status_code == 429: # Límite de tasa excedido
                print(f"Límite de tasa excedido (intento {attempt+1}/{max_retries}). Reintentando en {retry_delay} segundos...")
                time.sleep(retry_delay)
                retry_delay *= 2 # Backoff exponencial
            else:
                raise e # Relanzar otros errores de la API
        except Exception as e:
            print(f"Ocurrió un error inesperado: {e}")
            raise e
    raise Exception(f"Falló la obtención de respuesta después de {max_retries} intentos debido a límites de tasa.")


def ejecutar_demostracion_asistente(preguntas_ejemplo):
    for pregunta in preguntas_ejemplo:
        print(f"Pregunta: {pregunta}")
        print(f"Respuesta: {asistente_virtual_con_chromadb(pregunta)}\n")

######
pregunta = "¿Cuál es la experiencia de Pedro Ramirez en administracion de proyectos?"

print(f"Pregunta: {pregunta}")
print(f"Respuesta: {asistente_virtual_sincontexto(pregunta)}\n")

Pregunta: ¿Cuál es la experiencia de Pedro Ramirez en administracion de proyectos?
Respuesta: No dispongo de información sobre la experiencia de Pedro Ramírez en administración de proyectos.



Voy a generar los pares que voy a necesitar para correr el fine-tuning del modelo usando la informacion de la base de datos vectorial.

In [10]:
from datasets import Dataset
import random

# Esta funcion genera ejemplos para fine-tuning utilizando la base de datos vectorial

# Defino algunas preguntas que quiero que el modelo aprenda a responder
# Estas preguntas deben ser relevantes para el contenido del PDF

# Lista inicial de preguntas específicas
preguntas_iniciales = [
    "¿Cuál es el cargo de Pedro Ramirez?",
    "¿Qué certificaciones tiene Pedro en gestión de proyectos?",
    "¿Cuál es la experiencia de Pedro en inteligencia artificial?",
    "¿Cuál es el enfoque de Pedro en la implementación de soluciones tecnológicas?",
    "¿Cuáles son los objetivos a largo plazo de Pedro Ramirez?",
    "¿Qué idiomas habla Pedro?",
    "¿En qué año obtuvo Pedro su licenciatura en Ciencias de la Computación?",
    "¿Dónde realizó Pedro su Postgrado en Machine Learning y AI?",
    "¿Cuáles son las responsabilidades clave de Pedro como Director Asociado de Soluciones de Negocios Digitales para las Américas?",
    "¿Qué tipo de iniciativas ha liderado Pedro en inteligencia artificial?",
    "¿Qué logros profesionales tiene Pedro?",
    "¿Qué tecnologías domina Pedro?",
    "¿Cuál es la experiencia de Pedro en desarrollo de software?",
    "¿Ha trabajado Pedro en proyectos de automatización?",
    "¿Qué tipo de soluciones digitales ha implementado Pedro?",
    "¿En qué empresas ha trabajado Pedro Ramirez?",
    "¿Cuál es la metodología de trabajo preferida de Pedro?",
    "¿Qué herramientas de análisis de datos maneja Pedro?",
    "¿Tiene Pedro experiencia en ciberseguridad?",
    "¿Qué formación académica tiene Pedro en negocios?",
    "¿Ha ocupado Pedro cargos de liderazgo de equipos?",
    "¿Cuál es la opinión de Pedro sobre la transformación digital?",
    "¿Qué contribuciones ha hecho Pedro en el área de IA generativa?",
    "¿Cómo aborda Pedro la mejora de la eficiencia operativa?",
    "¿Ha participado Pedro en proyectos internacionales?",
    "¿Qué habilidades blandas destacaría Pedro?",
    "¿Cuál fue el título de su tesis de licenciatura?",
    "¿Qué experiencia tiene Pedro con sistemas de información?",
    "¿Cómo influyó su postgrado en Machine Learning en su carrera?",
    "¿Qué proyectos recientes ha liderado Pedro?"
]

# Usar solo las preguntas iniciales para el fine-tuning
preguntas_para_ft = list(preguntas_iniciales)

# Mezclar las preguntas para evitar sesgos en el orden de entrenamiento
random.shuffle(preguntas_para_ft)

# Defino una funcion para formatear los pares
def formatear_pares(pregunta, respuesta):
    mensajes = [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False)

pres_generadas = []
print(f"Generando {len(preguntas_para_ft)} pares de (pregunta, respuesta) para el fine-tuning...")
for pregunta_ft in preguntas_para_ft:
    try:
        # Usar la función RAG para obtener la respuesta basada en el contexto
        respuesta_ft = asistente_virtual_con_chromadb(pregunta_ft)
        pres_generadas.append((pregunta_ft, respuesta_ft))
        # print(f"- Pregunta: {pregunta_ft}\n  Respuesta generada: {respuesta_ft}\n") # Comentado para evitar output excesivo
    except Exception as e:
        print(f"Error al generar respuesta para '{pregunta_ft}': {e}")

# Formatear los ejemplos y crear el nuevo dataset
ejemplos_ft = []
for p, r in pres_generadas:
    ejemplos_ft.append({"texto": formatear_pares(p, r)})

dataset_ft = Dataset.from_list(ejemplos_ft)

print(f"\nNuevo Dataset para fine-tuning generado con {len(dataset_ft)} ejemplos a partir de la base de datos vectorial:")
display(dataset_ft)

Generando 30 pares de (pregunta, respuesta) para el fine-tuning...
Límite de tasa excedido (intento 1/3). Reintentando en 10 segundos...
Límite de tasa excedido (intento 1/3). Reintentando en 10 segundos...
Límite de tasa excedido (intento 2/3). Reintentando en 20 segundos...
Límite de tasa excedido (intento 3/3). Reintentando en 40 segundos...
Error al generar respuesta para '¿Cuál es la opinión de Pedro sobre la transformación digital?': Falló la obtención de respuesta después de 3 intentos debido a límites de tasa.
Límite de tasa excedido (intento 1/3). Reintentando en 10 segundos...
Límite de tasa excedido (intento 2/3). Reintentando en 20 segundos...
Límite de tasa excedido (intento 3/3). Reintentando en 40 segundos...
Error al generar respuesta para '¿Qué certificaciones tiene Pedro en gestión de proyectos?': Falló la obtención de respuesta después de 3 intentos debido a límites de tasa.
Límite de tasa excedido (intento 1/3). Reintentando en 10 segundos...
Límite de tasa excedido

Dataset({
    features: ['texto'],
    num_rows: 12
})

Ahora definamos una funcion para poder hacer una pregunta con el modelo sin fine-tuning.

In [11]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=200):
    mensajes = [{"role": "user", "content": pregunta}]
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()



In [12]:
# Probar el modelo base con el prompt de prueba antes de ajustarlo
prompt_prueba = "¿Qué certificaciones tiene Pedro en gestión de proyectos?"
respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

Pedro tiene el certificado de gestión del proyecto de la Asociación de Gestión de Proyectos (PGPM) de la Universidad de Navarra.


Como se puede ver, la respuesta es **incorrecta**.

Ahora puedes usar este `dataset_ft` generado dinámicamente para entrenar tu modelo LoRA, en lugar del `dataset` original hardcodeado. De esta manera, el modelo aprenderá a generar respuestas concisas y basadas en la información recuperada de tu base de conocimientos.

In [13]:
# Configurar LoRA (semilla fija para un resultado reproducible en la grabación)

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# Entrenar con LoRA

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=15,
    learning_rate=0.01,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
    bf16=False, # Desactivar bfloat16
    fp16=True,  # Activar float16 si es compatible con tu GPU
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset_ft,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss'] # último paso del entrenamiento
#perdida_final = resultado_entrenamiento.training_loss # promedio del entrenamiento
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## **PASO 3: DESPLEGAR EL MODELO AJUSTADO**

"Desplegar" en este contexto significa guardar el adaptador LoRA de forma reutilizable, no levantar un servidor. Con `save_pretrained` queda listo para volver a cargarlo en cualquier notebook sin repetir el entrenamiento; `push_to_hub` es opcional si quieres tenerlo disponible en tu cuenta de Hugging Face.

In [70]:
# Guardar el adaptador LoRA localmente

modelo_lora.save_pretrained("/content/modelo_ajustado_lora")
print("Adaptador LoRA guardado en /content/modelo_ajustado_lora")

# Opcional: subir el adaptador a tu cuenta de Hugging Face para reutilizarlo fuera de esta sesión
# modelo_lora.push_to_hub("tu-usuario/tinyllama-atencion-clientes-lora")

Adaptador LoRA guardado en /content/modelo_ajustado_lora


## **PASO 4: VERIFICACION DE RESPUESTAS CON RAG Y CON EL MODELO AJUSTADO**

Con la base de conocimiento indexada y el modelo ya ajustado, conectamos ambas piezas en una sola función: recupera el fragmento relevante, se lo entrega al modelo ajustado junto con la pregunta, y genera la respuesta final. Este es el mismo patrón que se espera construir en el Hackathon 1.

# **VALIDANDO CON RAG. **

Para validar la respuesta, usaremos la misma pregunta que se hizo en las primeras celdas.

In [20]:
def asistente_virtual_con_chromadb(pregunta):
    # Recuperar el fragmento más relevante de ChromaDB
    resultados_recuperados = consultar_coleccion_chroma(pregunta, coleccion_chroma, n_resultados=1)
    fragmento = resultados_recuperados['documents'][0][0] # Tomar el primer y más relevante fragmento

    prompt = f"""Responde la pregunta usando SOLO la informacion proporcionada a continuacion.
                 Si la informacion no responde la pregunta, no inventes ninguna respuesta, responde diciendo que no sabes la respuesta.

Informacion: {fragmento}

Pregunta: {pregunta}

Respuesta Concisa, sin caracteres extraños."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", # Usar el modelo Groq para la generación
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def ejecutar_demostracion_asistente(preguntas_ejemplo):
    for pregunta in preguntas_ejemplo:
        print(f"Pregunta: {pregunta}")
        print(f"Respuesta: {asistente_virtual_con_chromadb(pregunta)}\n")

pregunta = "¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?"

print(f"Pregunta: {pregunta}")
print(f"Respuesta: {asistente_virtual_sincontexto(pregunta)}\n")

Pregunta: ¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?
Respuesta: Pedro Ramírez tiene más de 8 años de experiencia liderando proyectos de tecnología y transformación digital, con certificaciones PMP y Scrum Master, y ha entregado con éxito más de 30 iniciativas en empresas multinacionales.



**VALIDANDO CON FINE-TUNING. **

Para validar la respuesta, usaremos la misma pregunta que se uso anteriormente.

In [76]:
# Probar el modelo base con el prompt de prueba antes de ajustarlo
prompt_prueba = "¿Qué certificaciones tiene Pedro en gestión de proyectos?"
respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

NoSidenoteSidenoteSidenote『 < < <||| < << < <> Traceback< <| <|< <<||<| <<< <>birSidenoteSidenote reactjs fragmentSidenoteSidenote²).SidenoteSidenotegerufen Traceback<|<<|¿ < <¿ <|¿||¿< <¿| <>burgerSidenoteSidenote partiellementSidenoteSidenote Leip『 <|ExSidenoteSidenote жовтSidenoteSidenote Список fragmentSidenote『||Ex жовтSidenote『< < |||a◄a◄es◄aSidenoteSidenote橋『 <<> Traceback|| de◄a жовтSidenote reactjses◄esSidenoteSidenote &=\ < < |< < .SidenoteSidenoteён◄aégl◄aicut < < . жовтSidenote²). жовтSidenotegerufen『 <>sprSidenoteSidenote Traceback<<<>bir жовтSidenote partiellement жовтSidenote Leip Traceback<>burger『 <¿<|Ex Leip『| <¿> Traceback < < > fragmentSidenote reactjs encuentra Traceback<¿ <<¿


### **COMPARACIÓN: Modelo LoRA ajustado vs. Asistente RAG**

In [43]:
def comparar_respuestas(pregunta_a_comparar):
    print(f"\n--- Pregunta: {pregunta_a_comparar} ---")

    # Respuesta del modelo LoRA ajustado
    respuesta_lora = generar_respuesta(modelo_lora, pregunta_a_comparar)
    print(f"\nRespuesta (Modelo LoRA Ajustado):\n{respuesta_lora}")

    # Respuesta del Asistente RAG (con modelo Groq grande)
    respuesta_rag = asistente_virtual_con_chromadb(pregunta_a_comparar)
    print(f"\nRespuesta (Asistente RAG):\n{respuesta_rag}")

# Ejemplo de uso con una de las preguntas de entrenamiento
comparar_respuestas("¿Qué certificaciones tiene Pedro en gestión de proyectos?")

# Puedes probar con otras preguntas aquí
# comparar_respuestas("¿Cuál es el cargo de Pedro Ramirez?")


--- Pregunta: ¿Qué certificaciones tiene Pedro en gestión de proyectos? ---

Respuesta (Modelo LoRA Ajustado):
Di < < < 


<<| < <| <  < <<|> < <re looking for the 
< < <a, and the  < 1.

 
  <|>

Respuesta (Asistente RAG):
Diploma de posgrado en Gestión de Proyectos (PMI)


# **CONCLUSIONES: **

1. Se observa que usando el metodo RAG que los resultados son inmediatos y se eliminan las aluciones.

2. En relacion a LoRA, la respuesta es sin sentido pero esto se debe a que, con el modelo LoRA se necesitan realizar un entrenamiento con un mayor numero de casos para poder afectar el comportamiento del modelo y obtener respuestas coherentes, seria necesario proporcionar cientos o miles de ejemplos para realmente ver un cambio en el model.
